# Dataset Statistics: Identification Counts & Coefficients of Variation

This notebook summarises two core quality metrics for the SCI proteomics dataset:

1. **Identification counts** — how many precursors / proteins are detected per sample.
2. **Coefficients of variation (CVs)** — per-feature reproducibility across samples within each cell-line condition.

Results are saved as publication-ready figures under `Figures/Figure_2/`.

---

## Imports

In [1]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
import anndata
import seaborn as sns
from matplotlib import pyplot as plt

# TODO: Fix ProteoBench dependency
from proteobench.io.parsing.parse_ion import _LOAD_FUNCTIONS

from SCI_py.data.parse import df_to_anndata, align_obs_across_anndata
from SCI_py.metadata.parse import FILENAME_CONVERTERS, get_institute

tqdm.pandas()

## 0. Path Configuration

Set up input/output directories relative to the notebook location. Both `SCI_DATA_DIR` and `SCI_FIGURES_DIR` environment variables can override the defaults, which is useful for running the notebook in automated pipelines or on different machines.

In [2]:
NOTEBOOK_DIR = Path().resolve()
ROOT_OUT      = NOTEBOOK_DIR / "../../Data"
ROOT_OUT      = ROOT_OUT.resolve()
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Allow user override via environment variable
ROOT_OUT = Path(os.environ.get("SCI_DATA_DIR", ROOT_OUT))

ROOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {ROOT_OUT}")

FIGURES_OUT = NOTEBOOK_DIR / "../../Figures"
FIGURES_OUT = FIGURES_OUT.resolve()
FIGURES_OUT.mkdir(parents=True, exist_ok=True)

# Overwrite with environment variable
FIGURES_OUT = Path(os.environ.get("SCI_FIGURES_DIR", FIGURES_OUT))
print(f"Figures directory: {FIGURES_OUT}")

ANNDATA_IN = os.path.join(
    ROOT_OUT,
    'AnnData'
)

Output directory: /home/sam/SCI/SCI/Data
Figures directory: /home/sam/SCI/SCI/Figures


## 1. Load Data

Load the pre-processed AnnData objects for **precursor-level** and **protein-level** quantification. Both objects share the same observation metadata (samples × morphological / experimental annotations) but differ in their feature space (`n_vars`).

In [3]:
adata_precursor = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor.h5ad')
)
adata_protein = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor.h5ad')
)

In [4]:
adata_precursor

AnnData object with n_obs × n_vars = 3114 × 72061
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type'
    var: 'PG.ProteinGroups', 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.iRTPredicted', 'FG.PrecWindowNumber', 'FG.CScore'
    layers: 'X_log2'

In [5]:
adata_protein

AnnData object with n_obs × n_vars = 3114 × 72061
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type'
    var: 'PG.ProteinGroups', 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.iRTPredicted', 'FG.PrecWindowNumber', 'FG.CScore'
    layers: 'X_log2'

## 2. Identifications

Count the number of non-missing precursors/proteins per sample and visualise the distribution as box plots. Two views are generated for each level:

- **Split** — one panel per institute/condition, useful for spotting site-specific outliers.
- **Combined** — all samples in a single panel for an overall summary.

In [ ]:
from SCI_py.plot.identifications import plot_counts_overview

In [6]:
adata_precursor.obs['n_precursors'] = np.sum(~np.isnan(adata_precursor.X), axis=1)
adata_protein.obs['n_proteins'] = np.sum(~np.isnan(adata_protein.X), axis=1)

In [ ]:
# Precursor boxplots for identifications, splitted view, and shown in one figure
plot_counts_overview(
    adata=adata_precursor,
    count_col='n_precursors',
    split_conditions=True,
    show_xlabels=False,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='counts_overview_precursors_SPLIT.png',
    show=False
)
plot_counts_overview(
    adata=adata_precursor,
    count_col='n_precursors',
    split_conditions=False,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='counts_overview_precursors.png',
    show=False
)

# Protein boxplots for identifications, splitted view, and shown in one figure
plot_counts_overview(
    adata=adata_protein,
    count_col='n_proteins',
    split_conditions=True,
    show_xlabels=False,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='counts_overview_proteins_SPLIT.png',
    show=False
)
plot_counts_overview(
    adata=adata_protein,
    count_col='n_proteins',
    split_conditions=False,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='counts_overview_proteins.png',
    show=False
)

/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


## 3. Coefficients of Variation (CVs)

Compute per-feature CVs (std / mean × 100) within each biological condition: **K562**, **HEK293T**, **QC - HeLa**, and **Site-specific QC**. CVs are computed at both protein and precursor level and stored in `adata.uns['CVs']`.

Box plots are generated for two proteome subsets:
- **Common proteome** (`common_proteome=True`) — features detected in ≥ `min_n` samples across *all* conditions.
- **All features** (`common_proteome=False`) — features detected in ≥ `min_n` samples within each condition independently.

In [ ]:
from SCI_py.data.compute import compute_cv_matrix
from SCI_py.plot.cvs import plot_cvs_overview

In [65]:
protein_cvs = compute_cv_matrix(
    adata=adata_protein,
    conditions=[
        'K562',
        'HEK293T',
        'QC - Hela',
        'Site specific QC'
    ]
)
precursor_cvs = compute_cv_matrix(
    adata=adata_precursor,
    conditions=[
        'K562',
        'HEK293T',
        'QC - Hela',
        'Site specific QC'
    ]
)

adata_protein.uns['CVs'] = protein_cvs
adata_precursor.uns['CVs'] = precursor_cvs

/home/sam/SCI/SCI/SCI_py/SCI_py/data/compute.py:13: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X_sub, axis=0)
/home/sam/miniconda3/envs/sci_env/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sam/SCI/SCI/SCI_py/SCI_py/data/compute.py:13: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X_sub, axis=0)
/home/sam/miniconda3/envs/sci_env/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [66]:
plot_cvs_overview(
    cvs=adata_protein.uns['CVs'],
    split_conditions=True,
    min_n=10,
    common_proteome=True,
    ylim=130,
    plot_type='box',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='CV_overview_proteins_common_box',
    show=False
)
plot_cvs_overview(
    cvs=adata_protein.uns['CVs'],
    split_conditions=True,
    min_n=10,
    common_proteome=False,
    ylim=130,
    plot_type='box',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='CV_overview_proteins_all_box',
    show=False
)

plot_cvs_overview(
    cvs=adata_precursor.uns['CVs'],
    split_conditions=True,
    min_n=10,
    common_proteome=True,
    ylim=130,
    plot_type='box',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='CV_overview_precursors_common_box',
    show=False
)
plot_cvs_overview(
    cvs=adata_precursor.uns['CVs'],
    split_conditions=True,
    min_n=10,
    common_proteome=False,
    ylim=130,
    plot_type='box',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_2'),
    filename='CV_overview_precursors_all_box',
    show=False
)

/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
/home/sam/SCI/SCI/SCI_py/SCI_py/plot/identifications.py:25: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
